<a href="https://colab.research.google.com/github/SahasransuAcharjya/Text-to-Image-with-XLP-Pipeline/blob/main/text2image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Compatible libraries

In [1]:
!pip install numpy==1.26.4 "huggingface_hub==0.25.2" "transformers==4.46.1" "diffusers==0.31.0" "accelerate==0.34.2" "gradio==4.44.1" onnxruntime-gpu insightface opencv-python peft safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 126.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.27.1
    Uninstalling huggingface-hub-0.27.1:
      Successfully uninstalled huggingface-hub-0.27.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.0
    Uninstalling diffusers-0.37.0:
      Successfully uninstalled diffusers-0

In [3]:
!pip install numpy==1.26.4 diffusers transformers accelerate peft safetensors onnxruntime-gpu insightface opencv-python gradio==4.44.1

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 140.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 16.1 MB/s eta 0:00:00
  Attempting un

In [1]:
# Install core AI libraries
!pip install -U diffusers transformers accelerate peft safetensors

# Install face extraction and processing tools
!pip install -q onnxruntime-gpu insightface opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 78.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 101.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


Backend Engine for the Model

In [1]:
# MUST BE AT THE VERY TOP: Fixes PyTorch memory fragmentation
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import gc
import traceback
from PIL import Image
from diffusers import AutoPipelineForText2Image, AutoPipelineForInpainting
from diffusers.utils import load_image
import gradio as gr

# ==========================================
# 1. The Fooocus Style & Resolution Libraries
# ==========================================
FOOOCUS_STYLES = {
    "None": {"prompt": "{prompt}", "negative_prompt": ""},
    "Cinematic": {
        "prompt": "cinematic still {prompt}. emotional, harmonious, vignette, highly detailed, high budget, bokeh, cinemascope, moody, epic, gorgeous, film grain, grainy",
        "negative_prompt": "anime, cartoon, graphic, text, painting, crayon, graphite, abstract, glitch, deformed, mutated, ugly, disfigured"
    },
    "Digital Art": {
        "prompt": "concept art {prompt}. digital artwork, illustrative, painterly, matte painting, highly detailed",
        "negative_prompt": "photo, photorealistic, realism, ugly"
    },
    "Photographic": {
        "prompt": "cinematic photo {prompt}. 35mm photograph, film, professional, 4k, highly detailed",
        "negative_prompt": "drawing, painting, crayon, sketch, graphite, impressionist, noisy, blurry, soft, deformed, ugly"
    },
    "Anime": {
        "prompt": "anime artwork {prompt}. anime style, key visual, vibrant, studio anime, highly detailed",
        "negative_prompt": "photo, deformed, black and white, realism, disfigured, low contrast"
    }
}

FOOOCUS_RESOLUTIONS = {
    "Square (1:1)": (1024, 1024),
    "Portrait (3:4)": (896, 1152),
    "Portrait (9:16)": (768, 1344),
    "Landscape (4:3)": (1152, 896),
    "Landscape (16:9)": (1344, 768)
}

# ==========================================
# 2. The Core Engine
# ==========================================
print("Building the AI Engine...")
class CustomImageEngine:
    def __init__(self):
        self.pipe_t2i = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        ).to("cuda")

        self.pipe_t2i.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
        self.pipe_inpaint = AutoPipelineForInpainting.from_pipe(self.pipe_t2i).to("cuda")

        # --- VRAM OPTIMIZATIONS ---
        # Slice VAE decoding (Prevents massive memory spike at the final step)
        self.pipe_t2i.vae.enable_tiling()
        self.pipe_t2i.vae.enable_slicing()
        self.pipe_inpaint.vae.enable_tiling()
        self.pipe_inpaint.vae.enable_slicing()

        # NOTE: Removed enable_attention_slicing() because it triggers the "tuple shape" library bug!

        print("Engine is built and ready in VRAM!")

    def _apply_style(self, prompt, negative_prompt, style_name):
        style_data = FOOOCUS_STYLES.get(style_name, FOOOCUS_STYLES["None"])
        final_prompt = style_data["prompt"].replace("{prompt}", prompt)
        final_negative = style_data["negative_prompt"]
        if negative_prompt:
            final_negative = f"{final_negative}, {negative_prompt}".strip(", ")
        return final_prompt, final_negative

    def _get_dimensions(self, aspect_ratio):
        return FOOOCUS_RESOLUTIONS.get(aspect_ratio, (1024, 1024))

    def generate(self, prompt, style, aspect_ratio, negative_prompt="", cfg_scale=7.5, steps=30):
        styled_prompt, styled_neg = self._apply_style(prompt, negative_prompt, style)
        width, height = self._get_dimensions(aspect_ratio)

        self.pipe_t2i.set_ip_adapter_scale(0.0)
        dummy = Image.new("RGB", (224, 224))

        img = self.pipe_t2i(
            prompt=styled_prompt,
            negative_prompt=styled_neg,
            ip_adapter_image=dummy,
            width=width,
            height=height,
            guidance_scale=cfg_scale,
            num_inference_steps=steps
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

    def generate_with_face(self, prompt, face_img_path, style, aspect_ratio, negative_prompt="", cfg_scale=7.5, steps=30):
        styled_prompt, styled_neg = self._apply_style(prompt, negative_prompt, style)
        width, height = self._get_dimensions(aspect_ratio)

        self.pipe_t2i.set_ip_adapter_scale(0.7)

        # --- THE REAL VRAM SAVER ---
        # Auto-resize face uploads to perfectly fit the IP-Adapter and prevent VRAM explosions
        face_image = load_image(face_img_path).convert("RGB").resize((224, 224))

        img = self.pipe_t2i(
            prompt=styled_prompt,
            ip_adapter_image=face_image,
            negative_prompt=styled_neg,
            width=width,
            height=height,
            guidance_scale=cfg_scale,
            num_inference_steps=steps
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

# Initialize models
engine = CustomImageEngine()

# ==========================================
# 3. The Gradio Web UI
# ==========================================
print("Launching Web Interface...")

def ui_generate(prompt, style, ratio, negative, cfg, steps):
    if not prompt: raise gr.Error("Please enter a prompt!")
    try:
        return engine.generate(prompt, style, ratio, negative, cfg, steps)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Generation failed: {str(e)}")

def ui_face_swap(prompt, face_img, style, ratio, negative, cfg, steps):
    if not prompt: raise gr.Error("Please enter a prompt!")
    if not face_img: raise gr.Error("Please upload a face image!")
    try:
        return engine.generate_with_face(prompt, face_img, style, ratio, negative, cfg, steps)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Face swap failed: {str(e)}")

style_choices = list(FOOOCUS_STYLES.keys())
ratio_choices = list(FOOOCUS_RESOLUTIONS.keys())

with gr.Blocks(theme=gr.themes.Soft()) as web_app:
    gr.Markdown("# 🎨 AI Image Studio")
    gr.Markdown("Your custom generation engine with Advanced Control.")

    with gr.Tabs():
        # --- TAB 1: Standard Generation ---
        with gr.TabItem("Text to Image"):
            with gr.Row():
                with gr.Column(scale=1):
                    prompt_in = gr.Textbox(label="Prompt", placeholder="A majestic avatar of Lord Vishnu standing in the cosmos...", lines=3)
                    with gr.Row():
                        style_in = gr.Dropdown(choices=style_choices, value="Digital Art", label="Image Style")
                        ratio_in = gr.Dropdown(choices=ratio_choices, value="Square (1:1)", label="Aspect Ratio")
                    neg_in = gr.Textbox(label="Negative Prompt", value="blurry, ugly, distorted")

                    with gr.Accordion("⚙️ Advanced Settings", open=False):
                        cfg_slider = gr.Slider(minimum=1.0, maximum=15.0, value=7.5, step=0.5, label="Prompt Adherence (CFG Scale)")
                        steps_slider = gr.Slider(minimum=10, maximum=60, value=30, step=1, label="Quality (Inference Steps)")

                    gen_btn = gr.Button("Generate Image", variant="primary")
                with gr.Column(scale=1):
                    img_out = gr.Image(label="Output")

            gen_btn.click(
                fn=ui_generate,
                inputs=[prompt_in, style_in, ratio_in, neg_in, cfg_slider, steps_slider],
                outputs=img_out
            )

        # --- TAB 2: Face Swapping ---
        with gr.TabItem("Face Swapping"):
            with gr.Row():
                with gr.Column(scale=1):
                    face_prompt_in = gr.Textbox(label="Prompt", placeholder="A portrait of an astronaut...", lines=3)
                    face_upload = gr.Image(label="Upload Face Reference", type="filepath")
                    with gr.Row():
                        face_style_in = gr.Dropdown(choices=style_choices, value="Photographic", label="Image Style")
                        face_ratio_in = gr.Dropdown(choices=ratio_choices, value="Portrait (3:4)", label="Aspect Ratio")
                    face_neg_in = gr.Textbox(label="Negative Prompt", value="blurry, distorted")

                    with gr.Accordion("⚙️ Advanced Settings", open=False):
                        face_cfg_slider = gr.Slider(minimum=1.0, maximum=15.0, value=7.5, step=0.5, label="Prompt Adherence (CFG Scale)")
                        face_steps_slider = gr.Slider(minimum=10, maximum=60, value=30, step=1, label="Quality (Inference Steps)")

                    face_btn = gr.Button("Generate with Face", variant="primary")
                with gr.Column(scale=1):
                    face_out = gr.Image(label="Output")

            face_btn.click(
                fn=ui_face_swap,
                inputs=[face_prompt_in, face_upload, face_style_in, face_ratio_in, face_neg_in, face_cfg_slider, face_steps_slider],
                outputs=face_out
            )

web_app.launch(share=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Building the AI Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: h94/IP-Adapter
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Engine is built and ready in VRAM!
Launching Web Interface...


/tmp/ipykernel_8279/1541225198.py:155: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as web_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://948d22eab52c879706.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
